# Fig. 1: Projected span and its growth at records

*Code cells: MIT licence (`LICENSE`). Explanatory text and the figure: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang).*

In [1]:
import sys, json, math
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import plot_main_figures as pmf
pd.set_option('display.precision', 8)

## What the figure shows

Fig. 1 is a concept figure. Panel (a) shows one conformation of an ideal wormlike chain (WLC) with
$L/p=6$, projected onto a plane that contains the measurement axis $\hat{\mathbf e}$. The span
$X_c$ is the distance between the two extreme planes normal to $\hat{\mathbf e}$; it is compared
with the projected end-to-end distance $|\mathbf R\cdot\hat{\mathbf e}|$. Panel (b) plots the
projected position $Z(s)=\mathbf r(s)\cdot\hat{\mathbf e}$ along the contour together with its
running maximum and minimum; the span grows only on the *record* segments, where $Z$ exceeds its
previous maximum or falls below its previous minimum. This is the pathwise content of the
first-return identity of the article.

**The conformation is illustrative only.** It carries no quantitative claim and is not used
anywhere else in the paper.

## How the conformation is generated

The tangent $\mathbf t(s)$ performs rotational diffusion on the unit sphere with
$\langle\mathbf t(s)\cdot\mathbf t(s')\rangle=e^{-|s-s'|/p}$. In reduced units ($p=1$) this is
simulated by small Euler steps: a Gaussian increment of variance $h$ in the tangent plane, followed by
renormalization to unit length. Positions follow by trapezoidal integration of the tangent. The
random seed is chosen deterministically by `choose_chain()`, which scans a fixed range of seeds for a
conformation whose span clearly exceeds the projected end-to-end distance and whose extrema lie in
the interior of the contour, so that the figure illustrates the point it is meant to make.

In [2]:
seed = pmf.choose_chain()
s, r = pmf.illustrative_chain(seed=seed)
z = r[:, 2]
print('seed =', seed)
print('span X_c         =', z.max() - z.min())
print('|R.e|            =', abs(z[-1] - z[0]))
print('ratio span/|R.e| =', (z.max() - z.min()) / abs(z[-1] - z[0]))

seed = 20261244
span X_c         = 2.0914573363087934
|R.e|            = 0.4977985202238736
ratio span/|R.e| = 4.20141332555229


## Records

The thick segments in panel (b) are the contour positions at which $Z$ equals its running maximum
$M(s)$ or running minimum $m(s)$. Their total length is the part of the contour that contributes to
the span.

In [3]:
M = np.maximum.accumulate(z); m = np.minimum.accumulate(z)
rec = (np.abs(z - M) < 1e-12) | (np.abs(z - m) < 1e-12)
print('fraction of contour on record segments:', rec.mean().round(3))

fraction of contour on record segments: 0.452


## Panel (c): the same trace read backward from a record

The point $s^*$ (diamond in panel b) lies on a growing record of the running minimum. Read backward
from $s^*$, the height of the trace above the record level, $Z(s^*-\sigma)-Z(s^*)$, starts with slope
$v=|U(s^*)|$ and stays positive for all $0<\sigma\le s^*$: the reversed contour has not returned to the
record plane. The first-return identity states that the growth rate of the mean span is half the probability of this
event, with the starting speed weighted by $v$ (density $2v$ on $0<v<1$).

In [4]:
z_star_i = np.argmin(np.abs(s - 4.582))
grad = np.gradient(z, s)
back = z[: z_star_i + 1][::-1] - z[z_star_i]
print('s* =', s[z_star_i], '  v =', -grad[z_star_i], '  min height above the record for t>0:', back[1:].min())

s* = 4.582   v = 0.35992265400508927   min height above the record for t>0: 0.0007743179665266053


## Panel (d): the Milne boundary layer

$\langle\Psi\rangle_u(y)=\tfrac12\int_{-1}^1\Psi(y,u)\,du$ is the angle-averaged solution of the stationary
Milne problem at angular order $m=128$ (`code/build_boundary_moments.py`). Its far field is
$\langle y-u+z_M\rangle_u=y+z_M$, which extrapolates to zero at $y=-z_M$, a distance $z_Mp$ beyond the
record plane. Near the record the profile lies below the far field because the tangents there are not
equilibrated; at the plane it equals $M_0/2$.

In [5]:
from build_boundary_moments import stationary_milne
sol = stationary_milne(128)
psi0 = 0.5*(sol.evaluate([0.0]) @ sol.weights)[0]
print('z_M(m=128) =', sol.z, '   <Psi>_u at the record =', psi0, '   far field there =', sol.z)

z_M(m=128) = 0.7182110465610224    <Psi>_u at the record = 0.5506777212674567    far field there = 0.7182110465610224


## Render the figure

In [6]:
fig, meta = pmf.figure1()
fig

<Figure size 677.165x275.591 with 4 Axes>